### Week 6 notebook - Decision Trees and Random Forests, Mod C Data Science Capstone Fall 2026
### By Shuqin Ouyang

# Overall notebook structure:

This week covers decision trees and random forests, run the same way on both datasets: the full feature set, the real 0/1 target, a stratified 80/20 split of the full data, and the same shuffled stratified 5-fold cross-validation scored on F1. Part 1 looks at entropy and information gain for a decision tree's first split, and checks how badly an unrestricted tree overfits. Part 2 tunes the three overfitting controls from the lesson (max depth, min samples per leaf, min samples per split) on a single tree, choosing the simplest setting within one standard deviation of the best. Part 3 compares a single tree against a random forest, with one extra row that isolates feature randomization. Part 4 tests the final tree and forest once on held-out data against each dataset's all-failure F1 baseline, and Part 5 looks at which features each model relies on. Like Week 5, this is a breadth week, so the goal is a robustness check on Weeks 1 to 4 rather than a deep dive on one model.

Note on setups across Weeks 4, 5 and 6: they differ, so raw test F1 is not comparable across weeks. Week 4 used a plain 80/20 split with default stratified 5-fold CV and no class weights, Week 5 used a stratified 1,000-row training and 250-row test subsample with default stratified 5-fold CV and no class weights, and Week 6 uses a stratified 80/20 split of the full data, shuffled stratified 5-fold CV and balanced class weights. Comparisons across weeks therefore rely on F1 against each dataset's all-failure baseline and on cross-validated results.


# Student dataset analysis starts here:

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, precision_score, recall_score

In [2]:
# shared settings for both datasets
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # same folds for every model

# the 3 overfitting controls from the lesson, tuned on a single decision tree:
# max_depth = max decisions in a row (None = no limit), min_samples_leaf = smallest
# group a leaf can hold, min_samples_split = smallest group that can still be split
PARAM_GRID = {'max_depth': [3, 6, 12, None], 'min_samples_leaf': [1, 10, 50], 'min_samples_split': [2, 20, 100]}

# entropy = the lesson's split measure, balanced = rare class counts more, bootstrap = bagging
tree_settings = dict(class_weight='balanced', criterion='entropy', random_state=42)
forest_settings = dict(bootstrap=True, n_jobs=-1, **tree_settings)
N_TREES_EXPLORE = 50   # while comparing
N_TREES_FINAL = 200    # final forest

Helper functions (used for both datasets):

In [3]:
def check_overfitting_f1(model, X_train, y_train, label):
    ''' same as Week 4, now with shared shuffled folds and the CV F1
    standard deviation, so differences can be judged against fold noise '''
    train_f1 = f1_score(y_train, model.predict(X_train))
    cv_f1 = cross_val_score(model, X_train, y_train, cv=CV, scoring='f1')
    print(f"{label}: train F1 = {train_f1:.4f}, CV F1 = {cv_f1.mean():.4f} +/- {cv_f1.std():.4f}, gap = {train_f1 - cv_f1.mean():.4f}")
    return train_f1, cv_f1.mean(), cv_f1.std()

In [4]:
def split_data(df, target):
    ''' stratified 80/20 split of the full data. No scaling this week,
    trees split on thresholds so feature size does not matter '''
    df = df.dropna()
    return train_test_split(df.drop(columns=[target]), df[target], test_size=0.2, stratify=df[target], random_state=42)

In [5]:
def root_split(tree, columns):
    ''' entropy and information gain of the tree's first decision:
    information gain = entropy before - weighted average entropy of the two
    branches. sklearn uses base 2 and the class weights, so with balanced
    weights the root starts at 1 bit (a 50/50 mix) '''
    t = tree.tree_
    left, right = t.children_left[0], t.children_right[0]
    after = (t.weighted_n_node_samples[left] * t.impurity[left] + t.weighted_n_node_samples[right] * t.impurity[right]) / t.weighted_n_node_samples[0]
    print(f"first split: {columns[t.feature[0]]} <= {t.threshold[0]:.3g}, entropy before = {t.impurity[0]:.3f}, "
          f"after = {after:.3f}, information gain = {t.impurity[0] - after:.3f}")

In [6]:
def tune_tree(X, y):
    ''' try every combination in PARAM_GRID on a single decision tree with
    5-fold CV. Chosen = simplest setting (shallowest, then biggest leaves and
    splits) within 1 CV std of the best, smaller differences are noise.
    Returns the table and the chosen setting '''
    search = GridSearchCV(DecisionTreeClassifier(**tree_settings), PARAM_GRID, scoring='f1', cv=CV,
                          return_train_score=True, n_jobs=-1).fit(X, y)
    table = pd.DataFrame(search.cv_results_['params'])
    table['train_F1'] = search.cv_results_['mean_train_score']
    table['cv_F1'] = search.cv_results_['mean_test_score']
    table['cv_std'] = search.cv_results_['std_test_score']
    table['gap'] = table['train_F1'] - table['cv_F1']
    table = table.round(3)

    top = table.loc[table['cv_F1'].idxmax()]
    close = table[table['cv_F1'] >= top['cv_F1'] - top['cv_std']]
    best = close.sort_values(['max_depth', 'min_samples_leaf', 'min_samples_split'], ascending=[True, False, False]).iloc[0]
    shape = {'max_depth': None if pd.isna(best['max_depth']) else int(best['max_depth']),
             'min_samples_leaf': int(best['min_samples_leaf']), 'min_samples_split': int(best['min_samples_split'])}
    print(f"{len(close)} of {len(table)} settings within 1 std of the best. Chosen: {shape}, CV F1 = {best['cv_F1']:.3f}")
    return table, shape

In [7]:
def compare_models(X, y, shape, label):
    ''' decision tree vs random forest, overfitting check on each (calls
    check_overfitting_f1). Rows 3 vs 4 isolate feature randomization (random
    subset of features at each split), rows 2 vs 3 the effect of combining
    many trees by voting '''
    models = {'1 tree, unrestricted': DecisionTreeClassifier(**tree_settings),
              '2 tree, tuned': DecisionTreeClassifier(**shape, **tree_settings),
              '3 forest, all features per split': RandomForestClassifier(n_estimators=N_TREES_EXPLORE, max_features=None, **shape, **forest_settings),
              '4 forest, random features per split': RandomForestClassifier(n_estimators=N_TREES_EXPLORE, max_features='sqrt', **shape, **forest_settings)}
    rows = {}
    for name, model in models.items():
        model.fit(X, y)
        train_f1, cv_f1, cv_std = check_overfitting_f1(model, X, y, f"{label} {name}")
        rows[name] = {'train_F1': train_f1, 'cv_F1': cv_f1, 'cv_std': cv_std, 'gap': train_f1 - cv_f1}
    return pd.DataFrame(rows).T.round(3)

In [8]:
def final_models(shape, X_train, y_train, X_test, y_test):
    ''' tuned tree and final forest, test once: F1, precision, recall, and the
    baseline F1 of predicting every row as a failure, 2 * rate / (1 + rate).
    Feature importance: built-in (how much each feature reduced entropy across
    the splits, training data) for both, plus permutation importance for the
    forest (shuffle one feature on the test set) '''
    tree = DecisionTreeClassifier(**shape, **tree_settings).fit(X_train, y_train)
    forest = RandomForestClassifier(n_estimators=N_TREES_FINAL, max_features='sqrt', **shape, **forest_settings).fit(X_train, y_train)
    rate = y_test.mean()
    results = pd.DataFrame({name: {'test_F1': f1_score(y_test, m.predict(X_test)), 'precision': precision_score(y_test, m.predict(X_test)),
                                   'recall': recall_score(y_test, m.predict(X_test)), 'F1 if all predicted failure': 2 * rate / (1 + rate)}
                            for name, m in [('tree', tree), ('forest', forest)]})
    perm = permutation_importance(forest, X_test, y_test, scoring='f1', n_repeats=3, random_state=42, n_jobs=-1)
    top10 = lambda values: pd.Series(values, index=X_test.columns).nlargest(10).round(4).reset_index()
    features = pd.concat({'tree built-in': top10(tree.feature_importances_), 'forest built-in': top10(forest.feature_importances_),
                          'forest permutation': top10(perm.importances_mean)}, axis=1)
    return tree, forest, results.round(3), features

Dataset 1 background quick overview:

The dataset 1 used is the Taiwanese Bankruptcy Prediction dataset from UCI, link as https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction.
It contains company financial indicators and a binary outcome for bankruptcy.

In [9]:
# install uci package so can load the data using package
# install once in terminal or notebook

!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# load dataset from UCI repository using dataset id
data_bankruptcy = fetch_ucirepo(id=572)

df_bankruptcy = pd.concat(
    [data_bankruptcy.data.features.copy(),
     data_bankruptcy.data.targets.copy()],
    axis=1
)

# remove leading/trailing spaces from column names once, here
df_bankruptcy.columns = df_bankruptcy.columns.str.strip()

df_bankruptcy.head()

'pip' is not recognized as an internal or external command,
operable program or batch file.


,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability,Bankrupt?
0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,...,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469,1
1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,0.781506,...,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794,1
2,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,...,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474,1
3,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,0.781241,...,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982,1
4,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,0.781550,...,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490,1


In [10]:
# show dataset info
df_bankruptcy.info()

<class 'pandas.DataFrame'>
RangeIndex: 6819 entries, 0 to 6818
Data columns (total 96 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   ROA(C) before interest and depreciation before interest  6819 non-null   float64
 1   ROA(A) before interest and % after tax                   6819 non-null   float64
 2   ROA(B) before interest and depreciation after tax        6819 non-null   float64
 3   Operating Gross Margin                                   6819 non-null   float64
 4   Realized Sales Gross Margin                              6819 non-null   float64
 5   Operating Profit Rate                                    6819 non-null   float64
 6   Pre-tax net Interest Rate                                6819 non-null   float64
 7   After-tax net Interest Rate                              6819 non-null   float64
 8   Non-industry income and expenditure/rev

Student markdown:

For this week I wanted to see how decision trees and random forests, which can pick up nonlinear patterns and interactions, handle the bankruptcy data compared with the linear models from earlier weeks. I use all 95 features and the real 0/1 target, with a stratified 80/20 split of the full data (5,455 training rows and 1,364 test rows, about 3.2% bankrupt). No scaling is needed here, since trees split on thresholds, so the size of a feature's values does not matter.


In [11]:
# stratified 80/20 split, full feature set, real 0/1 target, full training set
X_train_bankrupt, X_test_bankrupt, y_train_bankrupt, y_test_bankrupt = split_data(df_bankruptcy, 'Bankrupt?')
print(len(X_train_bankrupt), "train rows,", len(X_test_bankrupt), "test rows, failure rate", round(y_train_bankrupt.mean(), 4))

5455 train rows, 1364 test rows, failure rate 0.0323


In [12]:
# Part 1: entropy and information gain of the first split, and the overfitting check
# for an unrestricted decision tree (train F1 vs CV F1)
tree_full_bankrupt = DecisionTreeClassifier(**tree_settings).fit(X_train_bankrupt, y_train_bankrupt)
root_split(tree_full_bankrupt, X_train_bankrupt.columns)
check_overfitting_f1(tree_full_bankrupt, X_train_bankrupt, y_train_bankrupt, "Bankruptcy single tree, unrestricted")

first split: Continuous interest rate (after tax) <= 0.782, entropy before = 1.000, after = 0.685, information gain = 0.315
Bankruptcy single tree, unrestricted: train F1 = 1.0000, CV F1 = 0.3437 +/- 0.0462, gap = 0.6563


(1.0, np.float64(0.343694906428727), np.float64(0.04616259063101707))

Student markdown:

The unrestricted tree's first split is on Continuous interest rate (after tax) at 0.782. Entropy before the split is 1.000 (sklearn uses base 2, and with balanced class weights the root starts at an even 50/50 mix) and 0.685 after, so the information gain is 0.315. This matches the lesson, the tree picks the question that leaves the two branches purer. With no limit on depth, the tree fits the training data perfectly (train F1 = 1.000) but its CV F1 is only 0.344 +/- 0.046, a gap of 0.656. This is the lesson's overfitting example, a tree with one sample per leaf has basically memorized the training set.


In [13]:
# Part 2, tuning: max_depth, min_samples_leaf, min_samples_split on a single tree
tuning_bankrupt, shape_bankrupt = tune_tree(X_train_bankrupt, y_train_bankrupt)
tuning_bankrupt

6 of 36 settings within 1 std of the best. Chosen: {'max_depth': 12, 'min_samples_leaf': 1, 'min_samples_split': 20}, CV F1 = 0.324


,max_depth,min_samples_leaf,min_samples_split,train_F1,cv_F1,cv_std,gap
0,3.0,1,2,0.311,0.265,0.041,0.046
1,3.0,1,20,0.311,0.265,0.041,0.046
2,3.0,1,100,0.311,0.265,0.041,0.046
3,3.0,10,2,0.311,0.265,0.041,0.046
4,3.0,10,20,0.311,0.265,0.041,0.046
5,3.0,10,100,0.311,0.265,0.041,0.046
6,3.0,50,2,0.287,0.250,0.035,0.037
7,3.0,50,20,0.287,0.250,0.035,0.037
8,3.0,50,100,0.287,0.250,0.035,0.037
9,6.0,1,2,0.465,0.296,0.064,0.169


Student markdown:

Out of 36 settings, 6 were within one standard deviation of the best (best CV F1 = 0.344), and the simplest of those is max depth 12, min samples per leaf 1, min samples per split 20 (CV F1 = 0.324). What stood out is that bankruptcy prefers a fairly deep tree with very small leaves. Depth 3 is too shallow (CV F1 about 0.265, the underfitting side of the lesson), larger leaves hurt (leaf size 50 drops CV F1 to about 0.255), and min samples per split of 100 also hurts. I think this is because only about 176 training companies went bankrupt, so a leaf has to be small to isolate them. The two leaf and split controls also overlap, leaf 10 and split 20 give identical results, which makes sense since a leaf size of 10 already means a node needs at least 20 samples to be split.


In [14]:
# Part 3, decision tree vs random forest at the chosen setting
compare_bankrupt = compare_models(X_train_bankrupt, y_train_bankrupt, shape_bankrupt, "Bankruptcy")
compare_bankrupt

Bankruptcy 1 tree, unrestricted: train F1 = 1.0000, CV F1 = 0.3437 +/- 0.0462, gap = 0.6563
Bankruptcy 2 tree, tuned: train F1 = 0.7805, CV F1 = 0.3239 +/- 0.0715, gap = 0.4566
Bankruptcy 3 forest, all features per split: train F1 = 0.6862, CV F1 = 0.4152 +/- 0.0263, gap = 0.2710
Bankruptcy 4 forest, random features per split: train F1 = 0.6848, CV F1 = 0.4329 +/- 0.0452, gap = 0.2520


,train_F1,cv_F1,cv_std,gap
"1 tree, unrestricted",1.000,0.344,0.046,0.656
"2 tree, tuned",0.780,0.324,0.072,0.457
"3 forest, all features per split",0.686,0.415,0.026,0.271
"4 forest, random features per split",0.685,0.433,0.045,0.252


Student markdown:

The forest clearly beats a single tree here. CV F1 goes from 0.324 for the tuned tree to 0.415 for a forest with all features at each split and 0.433 for a forest with random features, and the train/CV gap drops from 0.457 to about 0.25 to 0.27. Most of the gain comes from combining many trees by voting (row 2 to row 3, +0.091). Feature randomization (row 3 to row 4) adds only +0.018, which is smaller than the fold-to-fold spread (0.026 to 0.045), so I can't say for sure it helped. The forest is also steadier across folds than the tuned tree (spread 0.026 to 0.045 compared to 0.072). A gap of 0.25 is still a real sign of overfitting, even for a forest. One caveat is that the depth and leaf settings were tuned on a single tree and reused for the forest, so the forest might prefer different ones.


In [15]:
# Part 4, final tree and forest, test once
tree_bankrupt, forest_bankrupt, results_bankrupt, features_bankrupt = final_models(shape_bankrupt, X_train_bankrupt, y_train_bankrupt, X_test_bankrupt, y_test_bankrupt)
results_bankrupt

,tree,forest
test_F1,0.407,0.478
precision,0.333,0.351
recall,0.523,0.750
F1 if all predicted failure,0.062,0.062


Student markdown:

On the test set the forest reached F1 = 0.478, precision = 0.351, recall = 0.750, compared to F1 = 0.407, precision = 0.333, recall = 0.523 for the single tree. Predicting every company as bankrupt would only give F1 = 0.062, so the forest is about 7.7 times better than that baseline. A recall of 0.750 means it caught about 33 of the roughly 44 bankrupt companies in the test set, much higher than Week 4's logistic regression (recall 0.177), although the split, features and class weighting are different, so this is not a controlled comparison. With only about 44 bankrupt companies in the test set, test F1 can move a lot from one split to another, so I trust the CV numbers (0.32 to 0.43) more than this single test result.


In [16]:
# Part 5, top 10 features: tree vs forest built-in importance, and forest permutation importance
features_bankrupt

tree built-in          \
                                     index       0   
0     Continuous interest rate (after tax)  0.3354   
1                   Interest Expense Ratio  0.1017   
2               Total debt/Total net worth  0.0980   
3  Persistent EPS in the Last Four Seasons  0.0532   
4                       Cash Turnover Rate  0.0348   
5                              Quick Ratio  0.0315   
6       Degree of Financial Leverage (DFL)  0.0300   
7             Operating Profit Growth Rate  0.0250   
8  Realized Sales Gross Profit Growth Rate  0.0248   
9              Current Liability to Assets  0.0245   

                           forest built-in          \
                                     index       0   
0                     Borrowing dependency  0.0573   
1  Persistent EPS in the Last Four Seasons  0.0513   
2               Total debt/Total net worth  0.0503   
3               Net Income to Total Assets  0.0436   
4     Continuous interest rate (after tax)  0.0349   
5        Retained Earnings to Total Assets  0.0330   
6                             Debt ratio %  0.0315   
7                      Equity to Liability  0.0303   
8                         Net worth/Assets  0.0299   
9    Net profit before tax/Paid-in capital  0.0278   

                                  forest permutation          
                                               index       0  
0                              Net Value Growth Rate  0.0191  
1                                        Quick Ratio  0.0102  
2                Operating Profit Per Share (Yuan ¥)  0.0063  
3                        After-tax net Interest Rate  0.0059  
4                                       Tax rate (A)  0.0051  
5                             Operating Gross Margin  0.0040  
6                        Realized Sales Gross Margin  0.0040  
7  Interest Coverage Ratio (Interest expense to E...  0.0040  
8                             Operating Expense Rate  0.0037  
9                    Net Worth Turnover Rate (times)  0.0037

Student markdown:

The single tree puts most of its importance on one feature, Continuous interest rate (after tax) at 0.335, followed by Interest Expense Ratio (0.102) and Total debt/Total net worth (0.098). The forest spreads importance across many features instead, its top feature (Borrowing dependency) is only 0.057. Many of the forest's top features are leverage or profitability ratios (Borrowing dependency, Total debt/Total net worth, Debt ratio %, Equity to Liability, Net Income to Total Assets, Persistent EPS), which lines up with earlier weeks, where Debt ratio % had the largest Week 4 coefficient and profitability ratios were already a risk signal. What was unexpected is that permutation importance gives a very different list (Net Value Growth Rate, Quick Ratio, Operating Profit Per Share) with tiny values, the largest is only 0.019. With about 44 bankrupt companies in the test set and many correlated ratios sharing credit, I would not trust permutation importance's ranking here. The built-in importance is computed on training data, so it looks steadier, but it also tends to favor continuous features.


Dataset 2 background quick overview:

The dataset 2 used is the Default of Credit Card Clients dataset from UCI, link as https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients.

It contains demographic information, credit limits, repayment history, bill statements, and previous payment records for Taiwanese credit card clients, along with a binary outcome indicating whether the client defaulted on the next payment.

In [17]:
# UCI dataset already decides features and targets already, but still need to look into details / verify to make sure all ok
data_credit = fetch_ucirepo(id=350)

df_credit = pd.concat(
    [data_credit.data.features.copy(),
     data_credit.data.targets.copy()],
    axis=1
)

df_credit.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [18]:
# columns rename for easier reading

df_credit = df_credit.rename(columns={

    "Y": "default_next_month",

    "X1": "credit_limit",
    "X2": "gender",
    "X3": "education",
    "X4": "marital_status",
    "X5": "age",

    "X6": "repay_2005_sep",
    "X7": "repay_2005_aug",
    "X8": "repay_2005_jul",
    "X9": "repay_2005_jun",
    "X10": "repay_2005_may",
    "X11": "repay_2005_apr",

    "X12": "bill_2005_sep",
    "X13": "bill_2005_aug",
    "X14": "bill_2005_jul",
    "X15": "bill_2005_jun",
    "X16": "bill_2005_may",
    "X17": "bill_2005_apr",

    "X18": "pay_2005_sep",
    "X19": "pay_2005_aug",
    "X20": "pay_2005_jul",
    "X21": "pay_2005_jun",
    "X22": "pay_2005_may",
    "X23": "pay_2005_apr"
})

df_credit.head()

,credit_limit,gender,education,marital_status,age,repay_2005_sep,repay_2005_aug,repay_2005_jul,repay_2005_jun,repay_2005_may,...,bill_2005_jun,bill_2005_may,bill_2005_apr,pay_2005_sep,pay_2005_aug,pay_2005_jul,pay_2005_jun,pay_2005_may,pay_2005_apr,default_next_month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [19]:
# show dataset info
df_credit.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   credit_limit        30000 non-null  int64
 1   gender              30000 non-null  int64
 2   education           30000 non-null  int64
 3   marital_status      30000 non-null  int64
 4   age                 30000 non-null  int64
 5   repay_2005_sep      30000 non-null  int64
 6   repay_2005_aug      30000 non-null  int64
 7   repay_2005_jul      30000 non-null  int64
 8   repay_2005_jun      30000 non-null  int64
 9   repay_2005_may      30000 non-null  int64
 10  repay_2005_apr      30000 non-null  int64
 11  bill_2005_sep       30000 non-null  int64
 12  bill_2005_aug       30000 non-null  int64
 13  bill_2005_jul       30000 non-null  int64
 14  bill_2005_jun       30000 non-null  int64
 15  bill_2005_may       30000 non-null  int64
 16  bill_2005_apr       30000 non-null  int64
 17  pay_

Student markdown:

Same approach as bankruptcy, on all 23 credit card features and the real 0/1 target, with a stratified 80/20 split of the full data (24,000 training rows and 6,000 test rows, about 22.1% default).


In [20]:
# stratified 80/20 split, full feature set, real 0/1 target, full training set
X_train_credit, X_test_credit, y_train_credit, y_test_credit = split_data(df_credit, 'default_next_month')
print(len(X_train_credit), "train rows,", len(X_test_credit), "test rows, failure rate", round(y_train_credit.mean(), 4))

24000 train rows, 6000 test rows, failure rate 0.2212


In [21]:
# Part 1: entropy and information gain of the first split, and the overfitting check
# for an unrestricted decision tree (train F1 vs CV F1)
tree_full_credit = DecisionTreeClassifier(**tree_settings).fit(X_train_credit, y_train_credit)
root_split(tree_full_credit, X_train_credit.columns)
check_overfitting_f1(tree_full_credit, X_train_credit, y_train_credit, "Credit card single tree, unrestricted")

first split: repay_2005_sep <= 0.5, entropy before = 1.000, after = 0.880, information gain = 0.120
Credit card single tree, unrestricted: train F1 = 0.9986, CV F1 = 0.3963 +/- 0.0090, gap = 0.6023


(0.9985892974701401,
 np.float64(0.39629505111421937),
 np.float64(0.009042778466792044))

Student markdown:

The first split is on repay_2005_sep at 0.5, with entropy going from 1.000 to 0.880, an information gain of 0.120. That is much smaller than bankruptcy's first split (0.315), so the first question alone separates the two groups less cleanly here, though this is only one split of one tree, so I would not read too much into it. The unrestricted tree overfits just like bankruptcy's, train F1 = 0.999 but CV F1 = 0.396 +/- 0.009, a gap of 0.602.


In [22]:
# Part 2, tuning: max_depth, min_samples_leaf, min_samples_split on a single tree
tuning_credit, shape_credit = tune_tree(X_train_credit, y_train_credit)
tuning_credit

18 of 36 settings within 1 std of the best. Chosen: {'max_depth': 3, 'min_samples_leaf': 50, 'min_samples_split': 100}, CV F1 = 0.521


,max_depth,min_samples_leaf,min_samples_split,train_F1,cv_F1,cv_std,gap
0,3.0,1,2,0.522,0.521,0.012,0.000
1,3.0,1,20,0.522,0.521,0.012,0.000
2,3.0,1,100,0.522,0.521,0.012,0.000
3,3.0,10,2,0.522,0.521,0.012,0.000
4,3.0,10,20,0.522,0.521,0.012,0.000
5,3.0,10,100,0.522,0.521,0.012,0.000
6,3.0,50,2,0.522,0.521,0.012,0.000
7,3.0,50,20,0.522,0.521,0.012,0.000
8,3.0,50,100,0.522,0.521,0.012,0.000
9,6.0,1,2,0.547,0.522,0.012,0.025


Student markdown:

Credit card behaved the opposite way from bankruptcy. 18 of 36 settings were within one standard deviation of the best, all of the depth 3 and depth 6 settings, and the simplest of those is max depth 3, min samples per leaf 50, min samples per split 100 (CV F1 = 0.521). Depth 3 and depth 6 give almost the same CV F1 (0.521 to 0.525), while deeper trees get worse (0.48 to 0.50 at depth 12, and as low as 0.396 with no limit). So credit card needs a simple tree where bankruptcy needed a deep one, and depth is the control that matters here, leaf size and split size barely change anything at depth 3 to 6. One caveat is that depth 3 is the shallowest value I tested, so the true best could be even simpler, a finer grid including depth 2 would be worth checking.


In [23]:
# Part 3, decision tree vs random forest at the chosen setting
compare_credit = compare_models(X_train_credit, y_train_credit, shape_credit, "Credit card")
compare_credit

Credit card 1 tree, unrestricted: train F1 = 0.9986, CV F1 = 0.3963 +/- 0.0090, gap = 0.6023
Credit card 2 tree, tuned: train F1 = 0.5213, CV F1 = 0.5212 +/- 0.0120, gap = 0.0002
Credit card 3 forest, all features per split: train F1 = 0.5218, CV F1 = 0.5217 +/- 0.0105, gap = 0.0001
Credit card 4 forest, random features per split: train F1 = 0.5386, CV F1 = 0.5382 +/- 0.0113, gap = 0.0003


,train_F1,cv_F1,cv_std,gap
"1 tree, unrestricted",0.999,0.396,0.009,0.602
"2 tree, tuned",0.521,0.521,0.012,0.000
"3 forest, all features per split",0.522,0.522,0.011,0.000
"4 forest, random features per split",0.539,0.538,0.011,0.000


Student markdown:

Here the forest adds very little. The tuned tree already has a train/CV gap of 0.000 (F1 = 0.521 for both), the forest with all features gives 0.522, and the forest with random features gives 0.538, a gain of about 0.017 over the tuned tree, which is roughly the size of the fold-to-fold spread (0.011 to 0.012). The big change is from the unrestricted tree (CV F1 = 0.396, gap 0.602) to the tuned tree, so limiting depth is what fixed the overfitting, not combining trees. This is the opposite of bankruptcy, where combining trees added about 0.09. Feature randomization again gave a small gain (+0.016) that is within the noise, similar to bankruptcy.


In [24]:
# Part 4, final tree and forest, test once
tree_credit, forest_credit, results_credit, features_credit = final_models(shape_credit, X_train_credit, y_train_credit, X_test_credit, y_test_credit)
results_credit

,tree,forest
test_F1,0.507,0.531
precision,0.450,0.499
recall,0.580,0.566
F1 if all predicted failure,0.362,0.362


Student markdown:

On the test set the forest reached F1 = 0.531, precision = 0.499, recall = 0.566, and the single tree reached F1 = 0.507, precision = 0.450, recall = 0.580. Predicting every customer as default would give F1 = 0.362, so the forest is only about 1.5 times better than that baseline, a much smaller gain than bankruptcy's 7.7 times. Week 4's logistic regression got test F1 = 0.351 on credit card, slightly below this 0.362 baseline, so trees look like a real improvement, although the split, features and class weighting differ, so this is not a controlled comparison. Test F1 (0.531) is also close to CV F1 (0.538), which fits credit card's stable picture, and with about 1,300 defaults in 6,000 test rows the test result is much less noisy than bankruptcy's.


In [25]:
# Part 5, top 10 features: tree vs forest built-in importance, and forest permutation importance
features_credit

tree built-in         forest built-in         forest permutation        
            index       0           index       0              index       0
0  repay_2005_sep  0.8322  repay_2005_sep  0.3147     repay_2005_sep  0.0965
1    pay_2005_aug  0.0743  repay_2005_aug  0.1474     repay_2005_aug  0.0213
2  repay_2005_jun  0.0413  repay_2005_jun  0.1145       pay_2005_apr  0.0039
3  repay_2005_aug  0.0224  repay_2005_jul  0.1105     repay_2005_jul  0.0038
4  repay_2005_jul  0.0223  repay_2005_apr  0.0872       credit_limit  0.0038
5   bill_2005_sep  0.0076  repay_2005_may  0.0600       pay_2005_aug  0.0024
6    credit_limit  0.0000    credit_limit  0.0336       pay_2005_sep  0.0021
7          gender  0.0000    pay_2005_aug  0.0291     repay_2005_jun  0.0019
8       education  0.0000    pay_2005_sep  0.0264     repay_2005_may  0.0017
9  marital_status  0.0000    pay_2005_jul  0.0168       pay_2005_may  0.0017

Student markdown:

repay_2005_sep is the top feature in all three lists, the tree's built-in importance (0.832), the forest's built-in importance (0.315) and the forest's permutation importance (0.097), which matches Week 4, where it had the largest coefficient. The six repayment status columns together hold about 83% of the forest's built-in importance, and the tree gives credit_limit, gender, education and marital_status an importance of 0.0000. The built-in and permutation lists agree on the top two features (repay_2005_sep and repay_2005_aug), unlike bankruptcy, where the lists disagreed almost completely. The tree puts nearly all its importance on one feature while the forest spreads it across several months, which is expected, since the forest's random feature subsets force it to use features other than the single best one.


Both datasets compared:

In [26]:
# both datasets side by side: tree vs forest overfitting check
pd.concat({'Bankruptcy': compare_bankrupt, 'Credit card': compare_credit}, axis=1)

Bankruptcy                       \
                                      train_F1  cv_F1 cv_std    gap   
1 tree, unrestricted                     1.000  0.344  0.046  0.656   
2 tree, tuned                            0.780  0.324  0.072  0.457   
3 forest, all features per split         0.686  0.415  0.026  0.271   
4 forest, random features per split      0.685  0.433  0.045  0.252   

                                    Credit card                       
                                       train_F1  cv_F1 cv_std    gap  
1 tree, unrestricted                      0.999  0.396  0.009  0.602  
2 tree, tuned                             0.521  0.521  0.012  0.000  
3 forest, all features per split          0.522  0.522  0.011  0.000  
4 forest, random features per split       0.539  0.538  0.011  0.000

Student markdown:

Side by side, the two datasets overfit in different ways. Both unrestricted trees memorize the training data (gaps of 0.656 and 0.602). After tuning, credit card's gap falls to 0.000 for both the tuned tree and the forest, while bankruptcy's stays at 0.457 for the tuned tree and 0.25 to 0.27 for the forests. Bankruptcy's CV F1 also moves more between folds (0.026 to 0.072, compared to about 0.011 for credit card). So credit card's models are simple and stable, while bankruptcy's need more complexity and are noisier, which is consistent with Week 4, where credit card's L1 selection was also more stable.


In [27]:
# both datasets side by side: final test results
pd.concat({'Bankruptcy': results_bankrupt, 'Credit card': results_credit}, axis=1)

Bankruptcy        Credit card       
                                  tree forest        tree forest
test_F1                          0.407  0.478       0.507  0.531
precision                        0.333  0.351       0.450  0.499
recall                           0.523  0.750       0.580  0.566
F1 if all predicted failure      0.062  0.062       0.362  0.362

Student markdown:

Raw F1 would suggest credit card is easier (forest F1 = 0.531 vs. 0.478), but F1 depends on how rare the failures are, so each dataset has to be judged against its own all-failure baseline. Bankruptcy's forest is about 7.7 times its baseline (0.478 vs. 0.062) and credit card's is about 1.5 times (0.531 vs. 0.362). So bankruptcy is harder in the sense of stability and overfitting, but the forest picks up much more signal relative to its baseline there, while credit card is stable but its F1 stays only moderately above baseline. Recall is also higher for bankruptcy's forest (0.750 vs. 0.566), while precision is lower (0.351 vs. 0.499).


Student markdown:

Conclusion:

**Overfitting:** Unrestricted trees overfit badly on both datasets (train F1 about 1.0, CV F1 of 0.344 and 0.396, gaps of 0.656 and 0.602). Tuning fixed this completely for credit card (gap 0.000 for both the tuned tree and the forest) but not for bankruptcy, where the tuned tree's gap is 0.457 and the forest's is still 0.25 to 0.27, even after combining trees.

**Metrics and results:** Bankruptcy chose max depth 12, min samples per leaf 1, min samples per split 20 (tuned tree CV F1 = 0.324), and the 200-tree forest reached test F1 = 0.478, precision = 0.351, recall = 0.750, against an all-failure baseline of 0.062. Credit card chose max depth 3, min samples per leaf 50, min samples per split 100 (tuned tree CV F1 = 0.521), and its forest reached test F1 = 0.531, precision = 0.499, recall = 0.566, against a baseline of 0.362. The first split's information gain was 0.315 for bankruptcy and 0.120 for credit card.

**Expected vs. unexpected:** I expected the unrestricted trees to overfit and the forest to reduce it, and I expected repay_2005_sep to be a top credit card feature after Week 4. What was unexpected is that bankruptcy preferred a deep tree with tiny leaves while credit card preferred the shallowest tree I tested, that combining trees added about 0.09 CV F1 for bankruptcy but almost nothing for credit card, that feature randomization's effect was within the noise on both datasets, and that permutation importance on bankruptcy disagreed with the built-in importance lists.

**EDA usefulness:** Earlier EDA suggested credit card's columns are more independently informative while bankruptcy's ratios are redundant and spread out. This week's importance lists agree with that, the repayment status columns hold about 83% of credit card's forest importance, while bankruptcy's importance is spread across many leverage and profitability ratios. EDA's scale and outlier findings did not matter this week, since trees do not need scaling.

**Sources:** This week's lesson content itself (entropy and information gain, the three overfitting controls, bagging and feature randomization, and feature importances), directly applied throughout this notebook rather than cited from an outside source.

**Bigger picture, research questions:** For the first research question, repay_2005_sep is again the top credit card signal, and bankruptcy's forest leans on leverage and profitability ratios, consistent with earlier weeks. For the second, relative to their baselines the forest gains much more on bankruptcy (7.7 times vs. 1.5 times), but bankruptcy's models overfit more and vary more across folds, so bankruptcy stays harder to model reliably even though a single F1 number alone would suggest otherwise. For the third, credit card's signal is stable across the tree, the forest, both importance types and Week 4, while bankruptcy's is less stable, the built-in lists share a few features (Total debt/Total net worth, Persistent EPS, Continuous interest rate) but permutation importance disagrees.

**How I know the conclusions are true:** These are grounded in cross-validated F1 with its fold-to-fold spread on the same 5 folds for every model, train/CV gaps, and test results judged against each dataset's own all-failure baseline. Caveats worth stating: bankruptcy's test set has only about 44 bankrupt companies, so its test numbers are noisy, the settings were tuned on a single tree and reused for the forest, credit card's chosen depth sits at the edge of the grid, and comparisons with Week 4 are not controlled since the split, features and class weighting differ.


Summary:

This week tested whether decision trees and random forests, which can capture nonlinear patterns and interactions, change the picture from Weeks 1 to 4. For credit card, the answer is mostly no. A shallow depth 3 tree already reached CV F1 = 0.521 with no overfitting, the forest added only about 0.017, and repay_2005_sep came out on top in every importance list, the same signal Week 4 found. For bankruptcy, trees behaved very differently. It needed a deep tree with tiny leaves, the forest added about 0.09 CV F1 over the tuned tree, and test F1 = 0.478 was about 7.7 times the all-failure baseline of 0.062, but it still overfit more (gap about 0.25) and varied more across folds, and its feature rankings were less stable, with built-in importance pointing to leverage and profitability ratios while permutation importance pointed somewhere else.

One thing worth connecting back to Week 4, where both datasets picked the identical C (0.359) from a coarse 10-value grid. This week's grid is also coarse (4 depths, 3 leaf sizes, 3 split sizes), and credit card's best depth landed on its edge, but the two datasets picked very different settings this time (12/1/20 vs. 3/50/100), so this kind of tuning can at least tell the two datasets apart, even though it does not by itself explain Week 4's identical C. Overall, Week 6 supports the earlier pattern, credit card's signal is concentrated and stable, bankruptcy's is spread across many ratios and less stable, though a single test set of about 44 bankrupt companies means the bankruptcy numbers should be read with care.
